# Õppetund 13 - Agendi mälestus


## Üldseadistused

See märkmik demonstreerib, kuidas luua reisibroneerimise agent koos **püsiva mäluga** kasutades **Microsoft Agent Framework'i** (MAF).

Sa õpid, kuidas erinevat tüüpi agendi mälu — töömälu, lühiajaline ja pikaajaline — mõjutavad seda, kuidas agent teavet vestluste jooksul säilitab ja kasutab.

**Nõuded:**
- Azure AI Foundry projekt koos juurutatud vestlusmudeliga (nt `gpt-4o-mini`).
- Azure CLI-sse sisse logitud — käivita terminalis käsk `az login`.
- `AZURE_AI_PROJECT_ENDPOINT` — sinu Azure AI Foundry projekti lõpp-punkt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — sinu juurutatud mudeli nimi.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Agendi mälutüübid

Tehisintellekti agendid saavad kasutada erinevaid mälutüüpe, millest igaühel on oma kindel eesmärk:

### Töömälu
Vestluse jutt ise — sõnumid, mis on vahetatud ühe seansi jooksul. Agent saab pöörduda tagasi varasemate sõnumite poole samas vestluses, et säilitada sidusust. MAF-is luuakse see **`agent.create_session()`** abil, mis tagastab `AgentSession`.

### Lühiajaline mälu
Teave, mis säilib ülesande või seansi kestel, kuid seda ei salvestata püsivalt. Näiteks võib agent koguda fakte mitme sammulise planeerimisvestluse jooksul ja kasutada neid lõpliku marsruudi koostamiseks.

### Pikaajaline mälu
Eelistused ja faktid, mis säilivad **seansside vahel**. Tagasi tulev kasutaja ei pea kordama oma toitumispiiranguid ega reisistiili. Pikaajaline mälu on tavaliselt toetatud välise andmebaasi, faili või vektorindeksi poolt ning selle kaudu esitatakse see agentidele tööriistade kaudu.


## Töömälu sessioonidega

Kõige lihtsam mälutüüp on vestluse sessioon. Kui edastate sama sessiooni (loodud `agent.create_session()` abil) järjepidevatele `agent.run()` kutsumistele, näeb agent kogu selle vestluse ajalugu ja suudab varasemaid üksikasju meenutada.

Loon reisibüroo ning demonstreerin töömäluga töötamist.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent mäletas eelarvet õigesti, sest mõlemad sõnumid jagavad sama seanssi. See on **töömälu** — see eksisteerib ainult seansi eluaja jooksul.

### Mis juhtub uue vestlusega?

Kui me loome **uue** seansi, siis agentil puudub mälu varasemast vestlusest:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Pikaajalise mälumustri

Kasutaja eelistuste **korduvaks meenutamiseks** sessioonide vahel vajame püsivat hoidlat, mis asub vestlussuuna väliselt. Agent pääseb sellele hoidlale ligi läbi **tööriistade** — funktsioonide, mida ta saab kutsuda informatsiooni salvestamiseks ja pärimiseks.

Allpool implementeerime lihtsa mälus oleva eelistuste hoidla (tootmiskeskkonnas võiks selle taga olla andmebaas või vektorindeks) ja ekspordime selle tööriistadena, mida agent saab kasutada.

### Arhitektuur
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Stsenaarium 1 — Esmakordne kasutaja broneerib aastapäeva reisi

Sarah külastab esimest korda. Agent peaks talletama tema eelistused tööriistade kaudu ja kasutama neid hotellide soovitamiseks.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Stsenaarium 2 — Sarah naaseb nädalate pärast

Sarah alustab **täiesti uut vestlust** (simuleerides uut seanssi). Töömälus pole midagi, kuid pikaajaline eelistuste salvesti sisaldab endiselt tema teavet. Agent peaks selle hankima ja kasutama soovituste isikupärastamiseks.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Kokkuvõte

Selles õppetükis õppisite tundma kolme liiki agendi mälusid ja kuidas neid Microsoft Agent Frameworki abil rakendada:

| Mälu tüüp | MAF mehhanism | Eluiga |
|---|---|---|
| **Töömälu** | `agent.create_session()` | Üks vestlus |
| **Lühiajaline** | Akumuleeritud kontekst lõimes | Üks ülesanne / seanss |
| **Pikaajaline** | Väliste andmeallikate kaudu ligipääsetav `@tool` funktsioonidega | Üle seansside |

### Peamised õppetunnid
1. **`agent.create_session()`** annab töömälu — agent näeb kogu vestluse ajalugu seansi jooksul.
2. **Uued seansid kaotavad konteksti** — ilma pikaajalise mäluta agent ei suuda meenutada eelnevaid vestlusi.
3. **`@tool` funktsioonid** loovad silla — võimaldades agendil salvestada ja taastada infot püsivast talletusest.
4. **Isikupärastamine paraneb aja jooksul** — mida rohkem eelistusi on salvestatud, seda paremad on agendi soovitused.

### Reaalmaailma rakendused
- **Klienditeenindus**: Mäleta kliendi ajalugu ja eelistusi
- **Isiklikud assistendid**: Säilita konteksti päevade või nädalate kaupa
- **Tervishoid**: Jälgi patsiendi infot ja eelistusi
- **E-kaubandus**: Isikupärastatud ostlemine ajaloopõhiselt

### Järgmised sammud
- Asenda mälu sisend-dictionary andmebaasi või vektorite talletusega (nt Azure AI Search)
- Lisa mälu aegumisaeg ajasensitiivsele infole
- Ehita mitme agendiga süsteeme jagatud mäluga
- Uuri Cognee märkmikku teadmistegraafiku toetatud mälu jaoks


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Vastutusest loobumine**:
See dokument on tõlgitud kasutades tehisintellekti tõlketeenust [Co-op Translator](https://github.com/Azure/co-op-translator). Kuigi püüdleme täpsuse poole, palun arvestage, et automaatsed tõlked võivad sisaldada vigu või ebatäpsusi. Originaaldokument selle emakeeles on käsitatav autoriteetse allikana. Kriitilise tähtsusega teabe puhul on soovitatav kasutada professionaalset inimtõlget. Me ei vastuta selle tõlke kasutamisest tulenevate arusaamatuste või valesti mõistmiste eest.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
